In [ ]:
# -*- coding: utf-8 -*-
"""
Полный скрипт дообучения cointegrated/rubert-tiny 
с автоматической подстройкой TrainingArguments, ранней остановкой и построением графиков.
Финальная версия с исправлением ошибки перезаписи переменной.
"""

import os
import glob
import csv
import json
import inspect
from collections import Counter
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)
from torch.nn import CrossEntropyLoss

# -----------------------
# Настройки
# -----------------------
INPUT_DIR = 'Data'
OUTPUT_DIR = './fine_tuned_rubert_tiny'
MODEL_NAME = "cointegrated/rubert-tiny"
MAX_LENGTH = 256
BATCH_SIZE = 16
EPOCHS = 10
LEARNING_RATE = 2e-5
GRAD_ACCUM_STEPS = 1
WEIGHT_DECAY = 0.01
SEED = 42
TEST_SIZE = 0.2

os.makedirs(OUTPUT_DIR, exist_ok=True)

# -----------------------
# Проверка устройства
# -----------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    acc = accuracy_score(labels, preds)
    f1_macro = f1_score(labels, preds, average="macro")
    return {"accuracy": acc, "f1_macro": f1_macro}

use_fp16 = torch.cuda.is_available()
print("FP16:", use_fp16)

# -----------------------
# Сбор файлов и данных
# -----------------------
files = glob.glob(os.path.join(INPUT_DIR, 'output_*.txt'))
if not files:
    raise FileNotFoundError(f"Файлы output_*.txt не найдены в {INPUT_DIR}")

data = []
unique_labels = set()

for file_path in files:
    base = os.path.basename(file_path)
    try:
        n_str = base.split('_')[1].split('.')[0]
        label = int(n_str)
        unique_labels.add(label)
    except (IndexError, ValueError):
        print(f"Пропуск файла с некорректным именем: {base}")

sorted_labels = sorted(list(unique_labels))
label_map = {orig: i for i, orig in enumerate(sorted_labels)}
num_classes = len(label_map)

print("\nНайдены классы:", sorted_labels)
print("Количество классов:", num_classes)

for file_path in files:
    base = os.path.basename(file_path)
    try:
        n_str = base.split('_')[1].split('.')[0]
        orig_label = int(n_str)
    except (IndexError, ValueError):
        continue

    if orig_label not in label_map:
        continue
        
    mapped_label = label_map[orig_label]

    with open(file_path, 'r', encoding='utf-8') as f:
        try:
            reader = csv.reader(f, delimiter=',', quotechar='"')
            texts = next(reader)
        except (StopIteration, csv.Error):
            print(f"Не удалось прочитать данные или файл пуст: {file_path}")
            continue

    for text in texts:
        text = text.strip()
        if text:
            data.append({"text": text, "label": mapped_label})

print(f"Всего примеров загружено: {len(data)}")

if not data:
    raise ValueError("Не найдено данных для обучения. Проверьте содержимое файлов.")

# -----------------------
# Разделение на обучающую и тестовую выборки
# -----------------------
all_labels = [d["label"] for d in data]
try:
    train_data, test_data = train_test_split(
        data, test_size=TEST_SIZE, random_state=SEED, stratify=all_labels
    )
except ValueError:
    print("Не удалось выполнить стратифицированное разделение, используется обычное.")
    train_data, test_data = train_test_split(
        data, test_size=TEST_SIZE, random_state=SEED
    )

print(f"Размер обучающей выборки: {len(train_data)}")
print(f"Размер тестовой выборки: {len(test_data)}")

train_dataset = Dataset.from_list(train_data)
test_dataset = Dataset.from_list(test_data)

# -----------------------
# Модель и токенайзер
# -----------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_classes)
model.to(device)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=MAX_LENGTH)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

tokenized_train = tokenized_train.rename_column("label", "labels")
# ИСПРАВЛЕНИЕ: Переименовываем столбец в tokenized_test, а не в test_dataset
tokenized_test = tokenized_test.rename_column("label", "labels")

final_columns = ['input_ids', 'attention_mask', 'labels']
tokenized_train.set_format(type='torch', columns=final_columns)
tokenized_test.set_format(type='torch', columns=final_columns)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# -----------------------
# Расчет весов классов
# -----------------------
train_labels_list = [d["label"] for d in train_data]
label_counts = Counter(train_labels_list)
total_samples = len(train_labels_list)
weights = []
for i in range(num_classes):
    count = label_counts.get(i, 0)
    weight = total_samples / (num_classes * count) if count > 0 else 0
    weights.append(weight)

class_weights = torch.tensor(weights, dtype=torch.float).to(device)
print("\nВеса классов для балансировки:", class_weights.cpu().numpy().tolist())

# -----------------------
# Кастомный Trainer с взвешенной функцией потерь
# -----------------------
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# -----------------------
# АВТОФИЛЬТРАЦИЯ TrainingArguments
# -----------------------
desired_args = {
    "output_dir": OUTPUT_DIR,
    "evaluation_strategy": "epoch",
    "eval_strategy": "epoch",
    "save_strategy": "epoch",           
    "learning_rate": LEARNING_RATE,
    "per_device_train_batch_size": BATCH_SIZE,
    "per_device_eval_batch_size": BATCH_SIZE,
    "num_train_epochs": EPOCHS,
    "weight_decay": WEIGHT_DECAY,
    "logging_dir": os.path.join(OUTPUT_DIR, "logs"),
    "logging_strategy": "steps",
    "logging_steps": 50,
    "load_best_model_at_end": True,
    "metric_for_best_model": "f1_macro",
    "greater_is_better": True,
    "fp16": use_fp16,
    "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
    "warmup_ratio": 0.1,
    "save_total_limit": 1,
    "seed": SEED,
    "report_to": "none",
}

sig = inspect.signature(TrainingArguments.__init__).parameters
supported_keys = set(sig.keys())
filtered_args = {k: v for k, v in desired_args.items() if k in supported_keys}

print("\nИспользуемые аргументы TrainingArguments:")
for key, val in filtered_args.items():
    print(f"- {key}: {val}")

training_args = TrainingArguments(**filtered_args)

# -----------------------
# Инициализация Trainer
# -----------------------
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

# -----------------------
# Обучение модели
# -----------------------
print("\n==== Начинаем обучение ====\n")
train_result = trainer.train()

trainer.model.save_pretrained(OUTPUT_DIR, safe_serialization=False)
tokenizer.save_pretrained(OUTPUT_DIR)
metrics = train_result.metrics
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)
trainer.save_state()
with open(os.path.join(OUTPUT_DIR, "label_map.json"), "w", encoding="utf-8") as f:
    json.dump({"label_map": label_map, "sorted_labels": sorted_labels}, f, ensure_ascii=False, indent=2)

# -----------------------
# Финальная оценка
# -----------------------
print("\n==== Оценка на тестовом наборе ====\n")
preds = trainer.predict(tokenized_test)
y_true = preds.label_ids
y_pred = np.argmax(preds.predictions, axis=-1)
report = classification_report(y_true, y_pred, target_names=[str(l) for l in sorted_labels], digits=4)
conf_matrix = confusion_matrix(y_true, y_pred)
print(report)
print("Матрица ошибок:")
print(conf_matrix)
with open(os.path.join(OUTPUT_DIR, "classification_report.txt"), "w", encoding="utf-8") as f:
    f.write(report)
    f.write("\nМатрица ошибок:\n")
    f.write(np.array2string(conf_matrix))

# -----------------------
# Построение графиков
# -----------------------
print("\n==== Построение графиков обучения ====\n")
log_history = trainer.state.log_history
df = pd.DataFrame(log_history)
train_logs = df[df['loss'].notna()].copy()
eval_logs = df[df['eval_loss'].notna()].copy()

if not train_logs.empty and not eval_logs.empty:
    plt.figure(figsize=(12, 6))
    plt.plot(train_logs['step'], train_logs['loss'], label='Training Loss')
    plt.plot(eval_logs['step'], eval_logs['eval_loss'], label='Validation Loss', marker='o')
    plt.title('График потерь (Training vs Validation Loss)')
    plt.xlabel('Шаги (Steps)')
    plt.ylabel('Потери (Loss)')
    plt.legend()
    plt.grid(True)
    loss_plot_path = os.path.join(OUTPUT_DIR, 'training_validation_loss.png')
    plt.savefig(loss_plot_path)
    print(f"График потерь сохранен в: {loss_plot_path}")
    plt.show()

    plt.figure(figsize=(12, 6))
    plt.plot(eval_logs['epoch'], eval_logs['eval_accuracy'], label='Validation Accuracy', marker='o')
    plt.plot(eval_logs['epoch'], eval_logs['eval_f1_macro'], label='Validation F1 Macro', marker='s')
    plt.title('График метрик на валидационном наборе')
    plt.xlabel('Эпоха (Epoch)')
    plt.ylabel('Значение')
    plt.legend()
    plt.grid(True)
    plt.xticks(np.arange(min(eval_logs['epoch']), max(eval_logs['epoch'])+1, 1.0))
    metrics_plot_path = os.path.join(OUTPUT_DIR, 'validation_metrics.png')
    plt.savefig(metrics_plot_path)
    print(f"График метрик сохранен в: {metrics_plot_path}")
    plt.show()
else:
    print("Недостаточно данных в логах для построения графиков.")

print("\n==== Обучение завершено ====")
print(f"Модель, токенайзер, метрики и графики сохранены в '{OUTPUT_DIR}'")

In [ ]:
# -*- coding: utf-8 -*-
"""
Полный скрипт дообучения DeepPavlov/rubert-base-cased
с аккуратным tqdm, логированием по эпохам, ранней остановкой,
взвешенной функцией потерь, безопасным сохранением и графиками.
"""
import os
import glob
import csv
import json
import inspect
from collections import Counter
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    TrainerCallback,
)
from torch.nn import CrossEntropyLoss

# -----------------------
# Настройки
# -----------------------
INPUT_DIR = 'Data'                  # Папка с файлами output_*.txt
OUTPUT_DIR = './fine_tuned_rubert_base'
MODEL_NAME = "DeepPavlov/rubert-base-cased"
MAX_LENGTH = 256
BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 2
EPOCHS = 10
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
SEED = 42
TEST_SIZE = 0.2

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "logs"), exist_ok=True)

# -----------------------
# Проверка устройства
# -----------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
use_fp16 = torch.cuda.is_available()
print("FP16:", use_fp16)

# -----------------------
# Функции метрик
# -----------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    acc = accuracy_score(labels, preds)
    f1_macro = f1_score(labels, preds, average="macro")
    return {"accuracy": acc, "f1_macro": f1_macro}

# -----------------------
# Сбор файлов и данных
# -----------------------
files = glob.glob(os.path.join(INPUT_DIR, 'output_*.txt'))
if not files:
    raise FileNotFoundError(f"Файлы output_*.txt не найдены в {INPUT_DIR}")

data = []
unique_labels = set()

for file_path in files:
    base = os.path.basename(file_path)
    try:
        n_str = base.split('_')[1].split('.')[0]
        label = int(n_str)
        unique_labels.add(label)
    except (IndexError, ValueError):
        print(f"Пропуск файла с некорректным именем: {base}")

sorted_labels = sorted(list(unique_labels))
label_map = {orig: i for i, orig in enumerate(sorted_labels)}
num_classes = len(label_map)

print("\nНайдены классы:", sorted_labels)
print("Количество классов:", num_classes)

for file_path in files:
    base = os.path.basename(file_path)
    try:
        n_str = base.split('_')[1].split('.')[0]
        orig_label = int(n_str)
    except (IndexError, ValueError):
        continue

    if orig_label not in label_map:
        continue
        
    mapped_label = label_map[orig_label]

    with open(file_path, 'r', encoding='utf-8') as f:
        try:
            reader = csv.reader(f, delimiter=',', quotechar='"')
            texts = next(reader)
        except (StopIteration, csv.Error):
            print(f"Не удалось прочитать данные или файл пуст: {file_path}")
            continue

    for text in texts:
        text = text.strip()
        if text:
            data.append({"text": text, "label": mapped_label})

print(f"Всего примеров загружено: {len(data)}")

if not data:
    raise ValueError("Не найдено данных для обучения. Проверьте содержимое файлов.")

# -----------------------
# Разделение на обучающую и тестовую выборки
# -----------------------
all_labels = [d["label"] for d in data]
try:
    train_data, test_data = train_test_split(
        data, test_size=TEST_SIZE, random_state=SEED, stratify=all_labels
    )
except ValueError:
    print("Не удалось выполнить стратифицированное разделение, используется обычное.")
    train_data, test_data = train_test_split(
        data, test_size=TEST_SIZE, random_state=SEED
    )

print(f"Размер обучающей выборки: {len(train_data)}")
print(f"Размер тестовой выборки: {len(test_data)}")

train_dataset = Dataset.from_list(train_data)
test_dataset = Dataset.from_list(test_data)

# -----------------------
# Модель и токенайзер
# -----------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_classes)
model.to(device)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=MAX_LENGTH)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

tokenized_train = tokenized_train.rename_column("label", "labels")
tokenized_test = tokenized_test.rename_column("label", "labels")

final_columns = ['input_ids', 'attention_mask', 'labels']
tokenized_train.set_format(type='torch', columns=final_columns)
tokenized_test.set_format(type='torch', columns=final_columns)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# -----------------------
# Расчет весов классов
# -----------------------
train_labels_list = [d["label"] for d in train_data]
label_counts = Counter(train_labels_list)
total_samples = len(train_labels_list)
weights = []
for i in range(num_classes):
    count = label_counts.get(i, 0)
    weight = total_samples / (num_classes * count) if count > 0 else 0
    weights.append(weight)

class_weights = torch.tensor(weights, dtype=torch.float).to(device)
print("\nВеса классов для балансировки:", class_weights.cpu().numpy().tolist())

# -----------------------
# Кастомный Trainer с взвешенной функцией потерь
# -----------------------
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# -----------------------
# АВТОФИЛЬТРАЦИЯ TrainingArguments
# -----------------------
desired_args = {
    "output_dir": OUTPUT_DIR,
    "evaluation_strategy": "epoch",
    "eval_strategy": "epoch",
    "save_strategy": "epoch",
    "learning_rate": LEARNING_RATE,
    "per_device_train_batch_size": BATCH_SIZE,
    "per_device_eval_batch_size": BATCH_SIZE,
    "num_train_epochs": EPOCHS,
    "weight_decay": WEIGHT_DECAY,
    "logging_dir": os.path.join(OUTPUT_DIR, "logs"),
    # Настройки для чистого вывода:
    "logging_strategy": "epoch",   # логируем только по эпохам (чтобы убрать "шум")
    "logging_steps": None,
    "disable_tqdm": True,          # отключаем встроенный tqdm (используем свой callback)
    "load_best_model_at_end": True,
    "metric_for_best_model": "f1_macro",
    "greater_is_better": True,
    "fp16": use_fp16,
    "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
    "warmup_ratio": 0.1,
    "save_total_limit": 1,
    "seed": SEED,
    "report_to": "none",
    # Важно: отключаем safetensors, чтобы Trainer сохранял стандартным torch.save
    "save_safetensors": False,
}

sig = inspect.signature(TrainingArguments.__init__).parameters
supported_keys = set(sig.keys())
filtered_args = {k: v for k, v in desired_args.items() if k in supported_keys}

print("\nИспользуемые аргументы TrainingArguments:")
for key, val in filtered_args.items():
    print(f"- {key}: {val}")

training_args = TrainingArguments(**filtered_args)

# -----------------------
# Callback'ы: чистый tqdm и красивый вывод метрик
# -----------------------
class SimpleTqdmCallback(TrainerCallback):
    """
    Простой прогресс-бар, который аккуратно обновляет global_step.
    Используется при disable_tqdm=True.
    """
    def __init__(self):
        self.pbar = None
        self.last_step = 0

    def on_train_begin(self, args: TrainingArguments, state, control, **kwargs):
        total = getattr(state, "max_steps", None) or args.max_steps
        # Если total неизвестен, создаем pbar без total (динамически)
        try:
            if total and total > 0:
                self.pbar = tqdm(total=total, desc="Training", ncols=120)
            else:
                self.pbar = tqdm(desc="Training", ncols=120)
        except Exception:
            self.pbar = tqdm(desc="Training", ncols=120)

    def on_step_end(self, args, state, control, **kwargs):
        try:
            cur = int(state.global_step)
            delta = cur - self.last_step
            if delta > 0 and self.pbar is not None:
                self.pbar.update(delta)
                self.last_step = cur
        except Exception:
            pass

    def on_train_end(self, args, state, control, **kwargs):
        if self.pbar:
            self.pbar.close()

class PrettyMetricsCallback(TrainerCallback):
    """
    Печатает аккуратную строку с eval-метриками при завершении оценки.
    """
    def on_evaluate(self, args, state, control, **kwargs):
        metrics = kwargs.get("metrics", None)
        if metrics is None:
            logs = [x for x in state.log_history if "eval_loss" in x]
            metrics = logs[-1] if logs else {}
        epoch = metrics.get("epoch", None)
        loss = metrics.get("eval_loss", None)
        acc = metrics.get("eval_accuracy", None)
        f1 = metrics.get("eval_f1_macro", None)
        runtime = metrics.get("eval_runtime", None)
        s = f"[Eval] "
        if epoch is not None:
            s += f"Epoch: {epoch:.0f} | "
        if loss is not None:
            s += f"loss: {loss:.4f} | "
        if acc is not None:
            s += f"acc: {acc:.4f} | "
        if f1 is not None:
            s += f"f1_macro: {f1:.4f} | "
        if runtime is not None:
            s += f"t: {runtime:.2f}s"
        print("\n" + s + "\n")

# -----------------------
# Инициализация Trainer
# -----------------------
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3),
               SimpleTqdmCallback(),
               PrettyMetricsCallback()]
)

# -----------------------
# Обучение модели
# -----------------------
print("\n==== Начинаем обучение ====\n")
train_result = trainer.train()

# -----------------------
# Безопасное финальное сохранение модели и токенайзера
# (приводим тензоры к contiguous() и переводим на CPU перед сохранением,
#  чтобы исключить возможные ошибки сериализации)
# -----------------------
print("\n==== Сохраняем модель и токенайзер (без safetensors) ====\n")
state_dict = trainer.model.state_dict()
new_state_dict = {}
for k, v in state_dict.items():
    if isinstance(v, torch.Tensor):
        if not v.is_contiguous():
            v = v.contiguous()
        v = v.cpu()
    new_state_dict[k] = v

# Сохраняем модель с модифицированным state_dict (используем safe_serialization=False)
trainer.model.save_pretrained(OUTPUT_DIR, state_dict=new_state_dict, safe_serialization=False)
tokenizer.save_pretrained(OUTPUT_DIR)

# Сохраняем label_map
with open(os.path.join(OUTPUT_DIR, "label_map.json"), "w", encoding="utf-8") as f:
    json.dump({"label_map": label_map, "sorted_labels": sorted_labels}, f, ensure_ascii=False, indent=2)

# Сохраняем результаты train
trainer.log_metrics("train", train_result.metrics)
trainer.save_metrics("train", train_result.metrics)
trainer.save_state()

# -----------------------
# Финальная оценка
# -----------------------
print("\n==== Оценка на тестовом наборе ====\n")
preds = trainer.predict(tokenized_test)
y_true = preds.label_ids
y_pred = np.argmax(preds.predictions, axis=-1)
report = classification_report(y_true, y_pred, target_names=[str(l) for l in sorted_labels], digits=4)
conf_matrix = confusion_matrix(y_true, y_pred)
print(report)
print("Матрица ошибок:")
print(conf_matrix)

with open(os.path.join(OUTPUT_DIR, "classification_report.txt"), "w", encoding="utf-8") as f:
    f.write(report)
    f.write("\nМатрица ошибок:\n")
    f.write(np.array2string(conf_matrix))

# -----------------------
# Построение графиков и красивая таблица метрик по эпохам
# -----------------------
print("\n==== Построение графиков обучения ====\n")
log_history = trainer.state.log_history
df = pd.DataFrame(log_history)

# Сохраним таблицу eval-показателей (если есть)
eval_logs = df[df['eval_loss'].notna()].copy()
if not eval_logs.empty:
    # Ренейм и отбор колонок
    cols_map = {
        "epoch": "epoch",
        "eval_loss": "loss",
        "eval_accuracy": "accuracy",
        "eval_f1_macro": "f1_macro",
        "eval_runtime": "eval_time"
    }
    for src, dst in cols_map.items():
        if src not in eval_logs.columns:
            eval_logs[src] = np.nan
    eval_df = eval_logs[list(cols_map.keys())].rename(columns=cols_map)

    # Печать аккуратной таблицы
    print("\n=== Eval metrics per epoch ===")
    print(eval_df.to_string(index=False, float_format='{:0.4f}'.format))

    # Сохраняем CSV
    csv_path = os.path.join(OUTPUT_DIR, "eval_metrics_per_epoch.csv")
    eval_df.to_csv(csv_path, index=False)
    print(f"\nEval metrics saved to: {csv_path}")

    # Рисуем графики (loss и метрики)
    try:
        plt.figure(figsize=(10, 5))
        plt.plot(eval_df['epoch'], eval_df['loss'], label='Validation Loss', marker='o')
        plt.title('Validation Loss per Epoch')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.grid(True)
        plt.xticks(eval_df['epoch'])
        loss_plot_path = os.path.join(OUTPUT_DIR, 'validation_loss_per_epoch.png')
        plt.savefig(loss_plot_path)
        plt.close()
        print(f"Validation loss plot saved to: {loss_plot_path}")

        plt.figure(figsize=(10, 5))
        plt.plot(eval_df['epoch'], eval_df['accuracy'], label='Validation Accuracy', marker='o')
        plt.plot(eval_df['epoch'], eval_df['f1_macro'], label='Validation F1 Macro', marker='s')
        plt.title('Validation Metrics per Epoch')
        plt.xlabel('Epoch')
        plt.ylabel('Value')
        plt.legend()
        plt.grid(True)
        plt.xticks(eval_df['epoch'])
        metrics_plot_path = os.path.join(OUTPUT_DIR, 'validation_metrics_per_epoch.png')
        plt.savefig(metrics_plot_path)
        plt.close()
        print(f"Validation metrics plot saved to: {metrics_plot_path}")
    except Exception as e:
        print("Не удалось построить графики:", e)
else:
    print("Недостаточно данных в логах для построения графиков.")

print("\n==== Обучение завершено ====")
print(f"Модель, токенайзер, метрики и графики сохранены в '{OUTPUT_DIR}'")


total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n==== Статистика параметров ====")
print(f"Всего параметров:     {total_params:,}")
print(f"Обучаемых параметров: {trainable_params:,}")
print(f"Процент обучаемых:    {100 * trainable_params / total_params:.2f}%")
print(f"==============================\n")